# 02 - Preprocessing

Goal: apply the cleaning rules and save the result.

The logic lives in `src/preprocessing.py`, not in this notebook.
The backend cannot import a notebook, so nothing important stays here.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src import config, lookups
from src.preprocessing import load_raw, clean

pd.set_option("display.max_columns", None)

In [2]:
df = load_raw(config.COLLISIONS_FILE)
print("Raw rows:", len(df))
print("Raw columns:", len(df.columns))

Raw rows: 513801
Raw columns: 44


In [3]:
sample = df[["weather_conditions", "light_conditions", "road_type", config.TARGET]].head(10).copy()

sample["weather"] = sample["weather_conditions"].map(lookups.WEATHER_CONDITIONS)
sample["light"] = sample["light_conditions"].map(lookups.LIGHT_CONDITIONS)
sample["road"] = sample["road_type"].map(lookups.ROAD_TYPE)
sample["severity"] = sample[config.TARGET].map(config.SEVERITY_LABELS)

sample[["weather", "light", "road", "severity"]]

,weather,light,road,severity
0,Fine no high winds,Darkness - lights lit,Single carriageway,Slight
1,Fine no high winds,Daylight,Single carriageway,Serious
2,Raining no high winds,Darkness - lights lit,Single carriageway,Slight
3,Fine no high winds,Daylight,Single carriageway,Fatal
4,Fine no high winds,Daylight,Single carriageway,Slight
5,Raining no high winds,Daylight,Dual carriageway,Slight
6,Raining no high winds,Darkness - lighting unknown,Single carriageway,Serious
7,Raining no high winds,Darkness - lighting unknown,Single carriageway,Slight
8,Fine no high winds,Daylight,Single carriageway,Slight
9,Fine no high winds,Darkness - lights lit,Single carriageway,Slight


In [4]:
df_clean = clean(df)

print("Rows before:  ", len(df))
print("Rows after:   ", len(df_clean))
print("Rows removed: ", len(df) - len(df_clean))
print()
print("Columns before:", len(df.columns))
print("Columns after: ", len(df_clean.columns))

Rows before:   513801
Rows after:    513698
Rows removed:  103

Columns before: 44
Columns after:  19


In [5]:
# 1. No -1 left anywhere
minus_one = (df_clean == -1).sum()
print("Columns still holding -1:", list(minus_one[minus_one > 0].index) or "none")

# 2. No blank values left
blanks = df_clean.isna().sum()
print("Columns still holding blanks:", list(blanks[blanks > 0].index) or "none")

# 3. How many rows ended up as Unknown in each column
print()
for col in ["weather_conditions", "road_surface_conditions", "road_type",
            "junction_detail", "second_road_class", "pedestrian_crossing",
            "carriageway_hazards", "trunk_road_flag"]:
    print(f"{col:<26} 99 = {(df_clean[col] == 99).sum():>6}")

# 4. Target balance must not have shifted
print()
print(df_clean[config.TARGET].value_counts(normalize=True).sort_index().round(4))

Columns still holding -1: none
Columns still holding blanks: none

weather_conditions         99 =  14566
road_surface_conditions    99 =  10230
road_type                  99 =  13053
junction_detail            99 =  21410
second_road_class          99 =  11603
pedestrian_crossing        99 =  23879
carriageway_hazards        99 =  19063
trunk_road_flag            99 =  36290

collision_severity
1    0.0147
2    0.2273
3    0.7580
Name: proportion, dtype: float64


In [6]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(config.CLEAN_FILE)
print("Saved to", config.CLEAN_FILE)

Saved to D:\ML_Final\roadsafe-ai\ml\data\processed\collisions_clean.parquet
